In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_openai import OpenAI, ChatOpenAI
from langchain.schema import HumanMessage
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

In [33]:
# Create LLM class
gemini = ChatGoogleGenerativeAI(
    model= "gemini-2.5-pro",
    temperature=1.0,
    max_retries=2,
    google_api_key=api_key,
)

In [34]:
llm1 = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
)

mistral = ChatHuggingFace(llm=llm1)

In [38]:
class BlogState(TypedDict):
    topic: str
    blogoutline: str
    final: str

In [51]:
def get_outline(state: BlogState) -> BlogState:
    print("Generating blog outline...")
    response = mistral.invoke([
        HumanMessage(content=f"Create a detailed blog outline on the topic: {state['topic']}")
    ])
    state['blogoutline'] = response.content
    print(state['blogoutline'])
    return state

In [ ]:
def get_blog(state: BlogState) -> BlogState:
    print("Generating final blog post...")
    blogoutline = state["blogoutline"]
    response = gemini.invoke([
        HumanMessage(content=f"Write a detailed blog post based on the following outline: {blogoutline}")
    ])
    state['final'] = response.content
    return state

In [53]:
#define graph
graph = StateGraph(BlogState)

In [54]:
# add nodes
graph.add_node("outlineblog", get_outline)
graph.add_node("finalblog", get_blog)

In [55]:
#add edges
graph.add_edge(START, "outlineblog")
graph.add_edge("outlineblog", "finalblog")
graph.add_edge("finalblog", END)

In [56]:
#compile graph
workflow = graph.compile()

In [58]:
#execute workflow
initial_state = BlogState(topic="The Future of Artificial Intelligence in Everyday Life", blogoutline="", final="")
final_state = workflow.invoke(initial_state)

Generating blog outline...
 Title: The Future of Artificial Intelligence in Everyday Life: Bringing Intelligent Technologies to Our Homes and Workplaces

I. Introduction
  A. Brief overview of Artificial Intelligence (AI) and its widespread use
  B. Importance of understanding the role of AI in every day life
  C. Preview of topics to be covered in the blog

II. AI in the Home
  A. Voice Assistants: Alexa, Siri, and Google Home
      1. Current capabilities
      2. Future potential
      3. Integration with smart home devices
  B. Personal Bots: Companions, caregivers, and educators
      1. Current applications
      2. Potential benefits and challenges
      3. Ethical considerations
  C. Smart Appliances: Energy efficiency, customization, and automation
      1. Current examples (fridges, washers, etc.)
      2. Future possibilities
      3. Making everyday life easier and more efficient

III. AI in the Workplace
  A. Automating Repetitive Tasks: Customer Service, Data Entry, etc.


In [59]:
final_state['topic']

'The Future of Artificial Intelligence in Everyday Life'

In [60]:
final_state['blogoutline']

' Title: The Future of Artificial Intelligence in Everyday Life: Bringing Intelligent Technologies to Our Homes and Workplaces\n\nI. Introduction\n  A. Brief overview of Artificial Intelligence (AI) and its widespread use\n  B. Importance of understanding the role of AI in every day life\n  C. Preview of topics to be covered in the blog\n\nII. AI in the Home\n  A. Voice Assistants: Alexa, Siri, and Google Home\n      1. Current capabilities\n      2. Future potential\n      3. Integration with smart home devices\n  B. Personal Bots: Companions, caregivers, and educators\n      1. Current applications\n      2. Potential benefits and challenges\n      3. Ethical considerations\n  C. Smart Appliances: Energy efficiency, customization, and automation\n      1. Current examples (fridges, washers, etc.)\n      2. Future possibilities\n      3. Making everyday life easier and more efficient\n\nIII. AI in the Workplace\n  A. Automating Repetitive Tasks: Customer Service, Data Entry, etc.\n   

In [61]:
final_state['final']

'Of course! Here is a detailed blog post written according to the provided outline.\n\n***\n\n### **The Future of Artificial Intelligence in Everyday Life: Bringing Intelligent Technologies to Our Homes and Workplaces**\n\nArtificial Intelligence is no longer the stuff of science fiction. It’s here, woven into the fabric of our daily routines—from the way we find our news to the music we listen to. But what we see today is just the tip of the iceberg. As AI continues to evolve at an exponential rate, its presence in our homes and workplaces is set to become even more integrated and transformative. Understanding this evolution is crucial for navigating the future we are all stepping into.\n\nThis post will take you on a journey through the current and future landscape of AI in our personal and professional lives. We will explore the smart assistants in our living rooms, the automated systems in our offices, and the creative potential of machines. Finally, we\'ll tackle the critical ethi